In [9]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import tensorflow as tf

In [2]:
cnn_train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    validation_split=0.2
)

cnn_train_generator = cnn_train_datagen.flow_from_directory(
    "Chest-Xray-2/chest_xray/train",
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

cnn_val_generator = cnn_train_datagen.flow_from_directory(
    "Chest-Xray-2/chest_xray/train",
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

cnn_test_datagen = ImageDataGenerator(rescale=1./255)

cnn_test_generator = cnn_test_datagen.flow_from_directory(
    "Chest-Xray-2/chest_xray/test",
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

Found 4187 images belonging to 2 classes.
Found 1045 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


In [3]:
resnet_train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,

    validation_split=0.2
)

resnet_train_generator = resnet_train_datagen.flow_from_directory(
    "Chest-Xray-2/chest_xray/train",
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='training',
    shuffle=True
)

resnet_val_generator = resnet_train_datagen.flow_from_directory(
    "Chest-Xray-2/chest_xray/train",
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation',
    shuffle=True
)

resnet_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

resnet_test_generator = resnet_test_datagen.flow_from_directory(
    "Chest-Xray-2/chest_xray/test",
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

Found 4187 images belonging to 2 classes.
Found 1045 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


In [4]:
cnn_model = load_model("custom_cnn_best_model.keras")
resnet_model = load_model("ResNet50_best_model.keras")

In [5]:
cnn_pred = cnn_model.predict(cnn_test_generator).flatten()
resnet_pred = resnet_model.predict(resnet_test_generator).flatten()

20/20 ━━━━━━━━━━━━━━━━━━━━ 23s 1s/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 27s 1s/step


In [6]:
ensemble_prob = (cnn_pred + resnet_pred) / 2

In [7]:
ensemble_prob = (0.3 * cnn_pred + 0.7 * resnet_pred)

In [8]:
ensemble_pred = (ensemble_prob > 0.5).astype("int32")

In [9]:
y_true = resnet_test_generator.classes

print(classification_report(y_true, ensemble_pred))

cm = confusion_matrix(y_true, ensemble_pred)
print(cm)

              precision    recall  f1-score   support

           0       0.91      0.85      0.88       234
           1       0.91      0.95      0.93       390

    accuracy                           0.91       624
   macro avg       0.91      0.90      0.90       624
weighted avg       0.91      0.91      0.91       624

[[199  35]
 [ 20 370]]


The ensemble improved overall reliability by combining the strengths of both models, achieving high recall while reducing false positives compared to individual models.

| Model                          | Accuracy    | Precision | Recall      | F1-Score    | AUC       | Key Strength     | Weakness             |
| ------------------------------ | ----------- | --------- | ----------- | ----------- | --------- | ---------------- | -------------------- |
| **Custom CNN**                 | 0.88        | **0.91**  | 0.89        | 0.88        | —         | Good precision   | Weak features        |
| **MobileNetV2**                | 0.87        | 0.84      | **🔥 0.97** | 0.86        | —         | Highest recall   | Many false positives |
| **ResNet50**                   | **🔥 0.91** | **0.92**  | 0.94        | **0.91**    | 0.957     | Best overall     | Slight FN            |
| **ResNet50 + CBAM**            | 0.89        | 0.89      | **🔥 0.95** | 0.89        | **0.957** | Best sensitivity | More FP              |
| **🔥 Ensemble (CNN + ResNet)** | **🔥 0.91** | **0.91**  | **🔥 0.95** | **🔥 0.91** | ~0.96     | **Best balance** | Slight complexity    |


The ensemble of Custom CNN and ResNet50 achieved the best balance between precision and recall, making it the most suitable model for pneumonia detection in a clinical setting.

---

## Complete Ensemble code

In [10]:
def get_img_array(img_path, size=(224, 224)):
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=size)
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    return preprocess_input(img_array)

In [11]:
def ensemble_predict(img_batch):
    
    cnn_input = img_batch / 255.0
    
    resnet_input = preprocess_input(img_batch.copy())
    
    cnn_pred = cnn_model.predict(cnn_input)
    resnet_pred = resnet_model.predict(resnet_input)
    
    # weighted ensemble
    final_pred = (0.3 * cnn_pred + 0.7 * resnet_pred)
    
    return final_pred

In [12]:
img = get_img_array("Chest-Xray-2/chest_xray/test/PNEUMONIA/person1_virus_6.jpeg")

prediction = ensemble_predict(img)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


In [13]:
print(prediction)

[[0.99848926]]


In [14]:
if prediction > 0.5:
    print("PNEUMONIA")
else:
    print("NORMAL")

PNEUMONIA


In [21]:
img2 = get_img_array("Chest-Xray-2/chest_xray/test/NORMAL/NORMAL2-IM-0023-0001.jpeg")

prediction = ensemble_predict(img2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step


In [22]:
print(prediction)

[[0.38337764]]


In [23]:
if prediction > 0.5:
    print("PNEUMONIA")
else:
    print("NORMAL")

NORMAL
